# Feature Engineering to reduce dimensionality

## Set Up

In [ ]:
import sys
import os

sys.path.append("../")
from src.config import BASE_PATH
from src.data_utils import get_data, get_feature_lists
import pandas as pd
import warnings

OUTCOME_DICT = {
    "outcome_bleed": get_data("outcome_bleed"),
    "outcome_pneumo": get_data("outcome_pneumo"),
    "outcome_reoper": get_data("outcome_reoper"),
    "outcome_ssi": get_data("outcome_ssi"),
    "outcome_serious": get_data("outcome_serious"),
    "outcome_any": get_data("outcome_any"),
}

# EDA (Correlation Exploration)

Findings:

- Optime ~correlated (>0.5) w/ other intra-op vars
- WNDINF, WTLOSS, DYSPNEA ~highly correlated (>=0.7)

In [ ]:
x_cols = [
    # Pre-op
    "SEX",  # Nominal
    "RACE_NEW",  # Nominal
    "ETHNICITY_HISPANIC",  # Binary
    "INOUT",  # Binary
    "Age",  # Numerical
    "URGENCY",  # Nominal
    "HEIGHT",  # Numerical
    "WEIGHT",  # Numerical
    "DIABETES",  # Binary
    "SMOKE",  # Binary
    "DYSPNEA",  # Binary
    "FNSTATUS2",
    "VENTILAT",
    "HXCOPD",
    "ASCITES",
    "HXCHF",
    "HYPERMED",
    "RENAFAIL",
    "DIALYSIS",
    "DISCANCR",
    "WNDINF",
    "STEROID",
    "WTLOSS",
    "BLEEDDIS",
    "TRANSFUS",
    "PRSEPIS",
    "PRALBUM",
    "PRWBC",
    "PRHCT",
    "PRPLATE",
    "ASACLAS",
    "OPERYR",
    # Intra-op
    "OPTIME",
    "Partial Glossectomy (Hemiglossectomy_Subtotal)",
    "Composite_Extended Glossectomy",
    "Total Glossectomy (Complete Tongue Removal)",
    "Excision of Tongue Lesions (Minor)",
    "Local_Regional Tissue Flaps for Oral Cavity Reconstruction",
    "Free Tissue Transfer (Microvascular Free Flaps) and Complex Flap Reconstruction",
    "Skin Autografts for Head and Neck Reconstruction",
    "Neck Dissection and Lymphadenectomy Procedures",
    "Alveolar Ridge and Gingival Procedures",
    "Mandibular Resection and Reconstruction Procedures",
    "Peripheral Nerve Repair and Neuroplasty",
    "Tracheostomy Procedures",
    "Gastrostomy and Esophageal Access Procedures",
    "Submandibular Gland Excision",
    "Parotid Gland Excision",
    "Laryngeal Resection and Reconstruction Procedures",
    "Pharyngeal Resection and Reconstruction Procedures",
    "Tonsillectomy and Tonsillar Region Procedures",
    "Malignant neoplasm",
]
x_cols_cap = [col.upper() for col in x_cols]

In [ ]:
df_raw = pd.read_excel(
    BASE_PATH / "data" / "processed" / "fully_cleaned_tongue_data.xlsx", index_col=0
)
df_dev = df_raw[df_raw["OPERYR"] != 2024]
df_dev = df_dev[x_cols_cap]
# can use any train set here arbitrarily
feature_lists = get_feature_lists(df_dev)
binary_cols = feature_lists["binary_cols"]
numerical_cols = feature_lists["numerical_cols"]
nominal_cols = feature_lists["nominal_cols"]
ordinal_cols = feature_lists["ordinal_cols"]

## Quick cleaning
replace_dict = {
    "SEX": {"male": "1", "female": "0"},
    "URGENCY": {"Urgent_Emergent": "1", "Elective": "0"},
    "ETHNICITY_HISPANIC": {"noUnknown": "0", "Yes": "1"},
    "ASACLAS": {
        "1-No Disturb": 1,
        "2-Mild Disturb": 2,
        "3-Severe Disturb": 3,
        "4/5-Life Threat/Moribund": 4,
    },
}
replace_w_binary_cols = [col for col in binary_cols if "Yes" in df_dev[col].unique()]
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", category=FutureWarning, message=".*Downcasting behavior.*"
    )
    df_dev = df_dev.replace(replace_dict).infer_objects(copy=False).copy()
    df_dev[replace_w_binary_cols] = (
        df_dev[replace_w_binary_cols]
        .replace({"Yes": 1, "No": 0})
        .infer_objects(copy=False)
        .copy()
    )

In [ ]:
def get_strong_pairs(corr_matrix, threshold=0.7, absolute=True):
    """
    Generated w/ LLM
    """
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            val = corr_matrix.iloc[i, j]
            if pd.isna(val):
                continue
            if absolute:
                condition = abs(val) >= threshold
            else:
                condition = val >= threshold
            if condition:
                pairs.append((cols[i], cols[j], val))
    # Sort by strength
    pairs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)

    return pd.DataFrame(pairs, columns=["Feature 1", "Feature 2", "Association"])

Numeric-like

In [ ]:
num_like_cols = binary_cols + ordinal_cols + numerical_cols

pearson_corr = df_dev[num_like_cols].corr(method="pearson")
spearman_corr = df_dev[num_like_cols].corr(method="spearman")

In [ ]:
strong_spearman = get_strong_pairs(spearman_corr, threshold=0.2)
display(strong_spearman.head(10))

In [ ]:
strong_pearson = get_strong_pairs(pearson_corr, threshold=0.2)
display(strong_pearson.head(10))

Nominal

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def cramers_v(x, y):
    tbl = pd.crosstab(x, y)
    chi2 = chi2_contingency(tbl)[0]
    n = tbl.to_numpy().sum()
    r, k = tbl.shape
    return np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

In [ ]:
nominal_corr = pd.DataFrame(index=nominal_cols, columns=nominal_cols, dtype=float)

for c1 in nominal_cols:
    for c2 in nominal_cols:
        if c1 == c2:
            nominal_corr.loc[c1, c2] = 1.0
        elif pd.isna(nominal_corr.loc[c1, c2]):
            v = cramers_v(df_dev[c1], df_dev[c2])
            nominal_corr.loc[c1, c2] = v
            nominal_corr.loc[c2, c1] = v

In [ ]:
strong_nominal = get_strong_pairs(nominal_corr, threshold=0.2, absolute=False)
display(strong_nominal.head(10))

Binary

In [ ]:
binary_corr = df_dev[binary_cols].corr(method="pearson")

In [ ]:
strong_binary = get_strong_pairs(binary_corr, threshold=0.2)
display(strong_nominal.head(10))

# Feature Selection

Game plan:

1) For each outcome x (w/ and w/o optime):
- Tune LightGBM w/ optuna--> score on avg precision
- Train LightGBM with optimal params on train set
- Record metrics on validation
- Run perm importance to get feature importance for CORRECT prediction

2) Compare validation performance accross (w/ and w/o optime)

3) Use perm rankings in whatever setting selected from above^

In [ ]:
import subprocess
import signal

### Set env
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_PATH)
## make cmd
outcome_list = list(OUTCOME_DICT.keys())
cmd = [
    "python",
    "src/feat_select.py",
    "--outcome_list",
    *outcome_list,
    "--num_optuna_trials",
    "50",
    "--num_perm_repeats",
    "300",
    "--optime_col_name",
    "OPTIME",
]
# call func
proc = subprocess.Popen(
    cmd,
    cwd=BASE_PATH,
    start_new_session=True,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

print("Controller PID:", proc.pid)

stdout, stderr = proc.communicate()

print("returncode:", proc.returncode)
print("STDOUT:\n", stdout)
print("STDERR:\n", stderr)

In [ ]:
proc.poll()  # monitor

In [ ]:
os.killpg(proc.pid, signal.SIGTERM)  # gracefull shtudown

In [ ]:
os.killpg(proc.pid, signal.SIGKILL)  # force kill

# Subset DFs w/ selected cols

In [ ]:
keep_cols = [
    "AGE",
    "ASACLAS",
    "BMI",
    "COMPOSITE_EXTENDED GLOSSECTOMY",
    "DIABETES",
    "DISCANCR",
    "EXCISION OF TONGUE LESIONS (MINOR)",
    "FNSTATUS2_Dependent",
    "FNSTATUS2_Independent",
    "FNSTATUS2_otherUnknown",
    "FREE TISSUE TRANSFER (MICROVASCULAR FREE FLAPS) AND COMPLEX FLAP RECONSTRUCTION",
    "HXCOPD",
    "HYPERMED",
    "INOUT",
    "LARYNGEAL RESECTION AND RECONSTRUCTION PROCEDURES",
    "LOCAL_REGIONAL TISSUE FLAPS FOR ORAL CAVITY RECONSTRUCTION",
    "MALIGNANT NEOPLASM_Malignant neoplasm of anterior two-thirds of tongue unspecified",
    "MALIGNANT NEOPLASM_Malignant neoplasm of base of tongue",
    "MALIGNANT NEOPLASM_Malignant neoplasm of border of tongue",
    "MALIGNANT NEOPLASM_Malignant neoplasm of junctional zone of tongue",
    "MALIGNANT NEOPLASM_Malignant neoplasm of lingual tonsil",
    "MALIGNANT NEOPLASM_Malignant neoplasm of surface of tongue",
    "MALIGNANT NEOPLASM_Malignant neoplasm of tongue unspecified",
    "NECK DISSECTION AND LYMPHADENECTOMY PROCEDURES",
    "OPERYR",
    "PARTIAL GLOSSECTOMY (HEMIGLOSSECTOMY_SUBTOTAL)",
    "PERIPHERAL NERVE REPAIR AND NEUROPLASTY",
    "PRALBUM",
    "PRHCT",
    "PRPLATE",
    "PRWBC",
    "RACE_NEW_Asian",
    "RACE_NEW_Black or African American",
    "RACE_NEW_White",
    "RACE_NEW_otherUnknown",
    "SEX",
    "SKIN AUTOGRAFTS FOR HEAD AND NECK RECONSTRUCTION",
    "SMOKE",
    "TOTAL GLOSSECTOMY (COMPLETE TONGUE REMOVAL)",
    "TRACHEOSTOMY PROCEDURES",
    "WNDINF_No",
    "WNDINF_Unknown(21-24)",
    "WNDINF_Yes",
    "WTLOSS_No",
    "WTLOSS_Unknown(21-24)",
    "WTLOSS_Yes",
]

In [ ]:
len(keep_cols)

In [ ]:
og_cols = OUTCOME_DICT["outcome_bleed"]["X_train"].columns
rem_cols = set(og_cols) - set(keep_cols)
list(rem_cols)

In [ ]:
from shutil import rmtree

for outcome_name, outcome_dict in OUTCOME_DICT.items():
    print(f"{outcome_name}...")
    export_dir = BASE_PATH / "data" / "processed_reduced" / outcome_name
    if export_dir.exists():
        rmtree(export_dir)
        print(f"\t Deleting dir at {export_dir}")
    export_dir.mkdir(exist_ok=True, parents=True)
    for set_type in ["train", "val", "test"]:
        y = outcome_dict[f"y_{set_type}"]
        X = outcome_dict[f"X_{set_type}"]
        X_sub = X[keep_cols].copy()
        assert X_sub.shape[1] == len(keep_cols)
        X_sub.to_parquet(export_dir / f"X_{set_type}.parquet")
        y.to_excel(export_dir / f"y_{set_type}.xlsx")